## Claud Desktop & MCP 생태계 입문
MCP docs 공식: https://modelcontextprotocol.io/docs/getting-started/intro
1. **Claud Desktop 앱**
    - 엔트로픽에서 서비스하는 claude 소프트웨어 앱 (API가 아닌 ChatGPT 서비스 앱과 같음)
    - claude desktop 앱에만 로컬에 설치된(사용자가 제작한) MCP 서버 프로그램과 통신할 수 있는 권한이 있음
    - 무료 claude 플랜을 사용하는 동안에는 세션 기반 사용량 제한이 있으며, 이 제한은 5시간마다 초기화. 또한 보낼 수 있는 메시지 수는 사용자 수요에 따라 가변적으로 제한됨
2. **claude_desktop_config.json 파일**
    - 기업이나 개인이 배포한 MCP서버(도구 모음집)를 claude에서 사용할 수 있게 해주는 개발자용 연결명세서 파일

In [1]:
!pip install mcp fastmcp

   ---------------------------------------- 0.0/705.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/705.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/705.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/705.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/705.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/705.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/705.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/705.5 kB ? eta -:--:--
   -------------- ------------------------- 262.1/705.5 kB ? eta -:--:--
   -------------- ------------------------- 262.1/705.5 kB ? eta -:--:--
   -------------- ------------------------- 262.1/705.5 kB ? eta -:--:--
   -------------- ------------------------- 262.1/705.5 kB ? eta -:--:--
   -------------- ------------------------- 262.1/705.5 kB ? eta -:--:--
   ---------------------------- --------- 524.3/705.5 kB 219.5 kB/s

### **1. 로컬 MCP 서버(도구모음) 만들기**
- MCP 서버는 모듈파일(.py)로 생성함
- langchain, langgraph에서 bind_tools를 사용해 도구를 바인딩(연결)했다면, 현재는 이를 py 파일로 외부로 빼둔 것.
- py 파일을 Claude Desktop이라는 앱(에이전트)이 통신해서 그 내부 도구들을 사용하게 됨
- **하나의 서버(.py파일)는 하나의 명확한 역할**만 하도록 설정하는 것이 리스크 관리 측면에서 좋음
- (하나의 서버에 모든 도구(기능)를 넣으면 해당 서버가 에러나면 에이전트가 다른 도구도 활용할 수 없음)

#### 1) 간단한 폴더 및 파일 검색 로컬 MCP 서버 개발

In [4]:
%%writefile mcpServer/my_mcp_server.py

import os
from mcp.server.fastmcp import FastMCP

# 서버 이름 설정 (시스템 내부적으로 사용되는 명칭)
mcp = FastMCP("PythonFileServer")


# 경로 내 파일 목록보기 도구
 # @mcp.tool() : MCP에서 함수를 도구로 만드는 명령
 # "~" 기호는 'C:/User/사용자PC명' 경로(홈 경로)를 뜻함
@mcp.tool()
def list_my_file(path: str = "~") -> list :
    """지정된 경로의 파일 목록을 반환합니다.
    경로가 없으면 기본적으로 사용자 홈 폴더를 조회합니다.
    """

    # expanduser: 경로 관련 기호들을 절대 경로로 바꿔주는 함수
    real_path = os.path.expanduser(path)

    try :
        # listdir : 경로 내 파일명 반환
        return os.listdir(real_path)
    except FileNotFoundError:
        return [f"에러: {real_path} 경로를 찾을 수 없습니다."]

# 파일 내용 읽기 도구
@mcp.tool()
def read_my_file(filepath: str) -> str :
    """지정된 절대 경로의 파일 (txt, csv, py 등) 내용을 읽어옵니다."""

    real_path = os.path.expanduser(filepath)

    # 허용할 확장자 입력
    allowed_extensions = ('.txt', '.csv', '.json', '.py', '.pdf', '.ipynb')
    
    # endswith : 문자열이 특정 글자로 끝나는지 확인
    if not real_path.lower().endswith(allowed_extensions) :
        return """에러: 이 도구는 txt, csv, json, py, pdf, ipynb 파일만 읽을 수 있습니다."""

    try :
        # 용량 검사: 파일이 5MB 이상이면 읽기 거부 (토큰 초과 방지)
        if os.path.getsize(real_path) > (5 * 1024 * 1024) :
            return """에러: 파일 크기가 너무 큽니다.(5MB 초과) LLM 토큰 제한을 위해 읽기를 거부합니다."""

        with open(real_path, "r", encoding='utf-8') as f :
            return f.read()

    # 파일 경로를 찾을 수 없는 경우
    except FileNotFoundError :
        return f"""에러: '{real_path}' 경로를 찾을 수 없습니다. list_my_files 도구로 경로를 먼저 확인하세요."""
    
    # 텍스트로 읽어올 수 없는 경우
    except UnicodeDecodeError :
        return f"""에러: '{real_path}'의 파일을 텍스트 형식으로 읽을 수 없습니다. (인코딩 혹인 바이너리 파일 문제)"""


# 파이썬에서 특정 파일을 터미널에서 직접 실행하는 코드
# claude desktop이 위 도구들을 실행할 때 해당 코드를 사용함
if __name__ == "__main__" :
    mcp.run()

Writing mcpServer/my_mcp_server.py


In [3]:
# expanduser 예시
import os
os.path.expanduser("~/Desktop")

'C:\\Users\\agnes/Desktop'

#### config.json 파일 내 코드 작성 방법
- config.json 파일을 통해 프로그램과 Claude가 표준 규격으로 대화를 나누기 시작.
- 이때부터 Claude desktop 채팅창에 '도구'로 나타남.
- 주의! 실제 config.json 파일 내에는 #으로 주석 달면 안 됨!!

In [6]:
# mcpServers: MCP 연결을 위한 고정된 문자열
{"mcpServers": {
    # 내 서버 별칭 (claude desktop 개발자 탭에 뜨는 명칭)
    "my-python-server": {
        # 'command'는 실행 프로그램명, 'args'는 지시사항으로 각각 작성된 코드를 터미널(CMD)에서 붙여서 실행하는 역할
        "command": "python",   # 로컬 python으로 명령 (절대경로를 써주는게 더 정확함)
        # py 파일 실행 경로 및 파일명 지정
        "args": [
            "C:/Users/agnes/용산5기/mcpServer/my_mcp_server.py"
        ],
        # 디폴트 경로 설정 (상대경로 관련 에러를 사전에 방지)
        "env": {
            "PYTHONPATH": "C:/Users/agnes/용산5기/mcpServer/"
        }
    }
}
}

{'mcpServers': {'my-python-server': {'command': 'python',
   'args': ['C:/Users/agnes/용산5기/mcpServer/my_mcp_server.py'],
   'env': {'PYTHONPATH': 'C:/Users/agnes/용산5기/mcpServer/'}}}}

In [7]:
# python 실행파일 경로 확인
import sys
print(sys.executable)

c:\Users\agnes\anaconda3\python.exe


- 위 config.json 파일은 우리가 직접 만든 MCP 서버를 claude desktop에 등록하는 파일
- command 항목에 **'uvx(python전용), npx(node.js전용)** 라는 실행 명령을 사용하면 다른 개발자들이 만들어둔 MCP 서버의 코드를 잠시 빌려와 실행 가능
    - **uvx**를 사용하면 전 세계 파이썬 개발자들의 공식 저장소인 `PyPI(Python Package Index)`라는 거대한 서버에서 코드를 가져와 실행하라는 뜻
    - **npx**를 사용하면 전 세계 자바스크립트 개발자들의 공식 저장소인 `npm(Node Package Manager)`라는 곳에서 실행
    - 이렇게 실행명령이 나눠져있는 이유는 엔트로픽 개발팀의 주력 언어가 node.js였고, Claude desktop 앱 자체도 node.js 기반으로 만들어졌기 때문 (파이썬은 추후에 기능 추가한 것)
    - `node.js` : 웹 페이지에 동적인 기능을 추가하기 위해 사용하는 자바스크립트 언어를 활용하여 서버를 만들거나 일반 프로그램을 작성할 수 있게 해주는 프로그램
    - 또한 **Smithery**나 **MCP Directory**와 같은 개방형 MCP 스토어에서 원하는 서버를 검색해서 install 하면 내 config.json 파일에 자동으로 설정 코드를 넣어 활용 가능

#### 엔트로픽 공식 MCP 서버 종류
일부 공식 서버들을 claude desktop의 확장프로그램에서 바로 붙여서 사용할 수 있지만, 세부사항 설정이나 커스터마이징된 도구를 사용하는데에는 제약이 따름 (claude desktop이 아닌 우리가 직접 에이전트를 구현할 때 사용하면 좋음)

#### 깃허브 사이트: https://github.com/modelcontextprotocol/servers
 - Filesystem: 폴더 및 파일 읽기, 생성 및 수정, 이동 및 관리 작업(npx 전용)
 - Puppeteer: 웹 브라우저 제어(브라우저 창 열기, 클릭 등 동적 크롤링 기능)
 - Fetch: url로 웹 검색 (웹 관련 서버 중 상대적으로 가벼운 구동)
 - SQLite: 데이터베이스 연동
 - Time: 로컬 시간 및 날짜
 - Google Maps: 장소 검색
 - Google Drive / Slack / GitHub: 협업 툴 사용

In [ ]:
{"mcpServers": {
    "my-python-server": {
        "command": "c:/Users/agnes/anaconda3/python.exe",
        "args": [
            "C:/Users/agnes/용산5기/mcpServer/my_mcp_server.py"
        ],
        "env": {
            "PYTHONPATH": "C:/Users/agnes/용산5기/mcpServer/"
        }
    }
}
}

### **2. MCP 공식 웹 크롤링 서버(Fetch) 사용해보기**
- url로 웹 크롤링을 수행하는 MCP 공식 서버
- (공식서버는 엔트로픽에서 해당 MCP서버를 외부 인터넷에 공개해두고 오픈소스로 활용할 수 있게 만들어뒀기 때문에 `API 호출이 필요 없음`)
- Claude Desktop 혹은 Claude API에서 url 을 주고 크롤링 수행을 시키면, AI 크롤링을 감지하여 접근 거부되거나 화면에 보이는 수많은 텍스트를 다 가져와서 오히려 성능이 좋지 않을 수 있는데, Fetch를 사용하면 관련 작업들을 무리없이 수행하게 만들 수 있음
- Claude Desktop에도 기본적으로 인터넷 검색 기능이 포함되어 있지만, 주체가 엔트로픽 본사 서버이므로 퍼블릭 웹 페이지 외 `AI봇 크롤링을 막아둔 사이트`나 `사내 폐쇄망`에는 접근이 불가할 가능성이 높음

In [8]:
!pip install uv

   ---------------------------------------- 0.0/25.2 MB ? eta -:--:--
    --------------------------------------- 0.5/25.2 MB 7.5 MB/s eta 0:00:04
   -- ------------------------------------- 1.6/25.2 MB 5.3 MB/s eta 0:00:05
   ---- ----------------------------------- 2.6/25.2 MB 5.5 MB/s eta 0:00:05
   ------ --------------------------------- 3.9/25.2 MB 5.6 MB/s eta 0:00:04
   ------- -------------------------------- 5.0/25.2 MB 5.4 MB/s eta 0:00:04
   ---------- ----------------------------- 6.6/25.2 MB 5.6 MB/s eta 0:00:04
   ------------ --------------------------- 7.9/25.2 MB 5.6 MB/s eta 0:00:04
   -------------- ------------------------- 8.9/25.2 MB 5.7 MB/s eta 0:00:03
   ---------------- ----------------------- 10.2/25.2 MB 5.7 MB/s eta 0:00:03
   ------------------ --------------------- 11.5/25.2 MB 5.7 MB/s eta 0:00:03
   ------------------- -------------------- 12.6/25.2 MB 5.6 MB/s eta 0:00:03
   --------------------- ------------------ 13.6/25.2 MB 5.6 MB/s eta 0:00:03
  

In [ ]:
import shutil
print(shutil.which("uvx"))  # 실행파일이 어디있는지 경로

C:\Users\agnes\anaconda3\Scripts\uvx.EXE


#### mcp-server-fetch 서버 구동 json 코드

In [ ]:
{
    "mcpServer": {
    "fetch":{
        "command": "C:/Users/agnes/anaconda3/Scripts/uvx.exe",
        "args": ["mcp-server-fetch"]
    }
}
}